# Multi-tier tower twin (replayable)

First spill photo → objects (length, width) → which filter trays the Partabot arm mounts in the active tiers → emulated arm run that fails closed.

**Swap the data:** set `OBJECTS` to an `objects.jsonl` from the vision lane (or BugPicker). Geometry in `baebugs/config/tower.yaml` is PLACEHOLDER until the tower is measured; filter sizes are nominal.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from baebugs import tower_twin as TT
OBJECTS = None   # e.g. 'scans/run1/objects.jsonl'
objs = [json.loads(l) for l in open(OBJECTS)] if OBJECTS else TT.demo_objects()
print(len(objs), 'objects')

285 objects


## Plan with 2 active tiers (rack + kit), then kit trays only

In [2]:
cfg = TT.load_yaml('tower.yaml')
inv = TT.tower_inventory()
full = TT.plan_tiers(objs, inv, cfg, k=2)
kit = TT.plan_tiers(objs, [f for f in inv if f['source']=='kit'], cfg, k=2)
for name, p in (('rack+kit', full), ('kit only', kit)):
    b = p['best']; print(f"{name:9s} {b['tiers']} purity {b['purity']} uncertainty {b['uncertainty']}")
    for f in b['fractions']: print('   ', f)

rack+kit  ['SS-SLOT-3.2x19', 'HEX-2'] purity 0.607 uncertainty 0.0436
    {'large': 8.0, 'medium': 0.0, 'small': 0.0, 'tiny': 0.0, 'xlarge': 22.0}
    {'large': 0.0, 'medium': 43.0, 'small': 55.0, 'tiny': 0.0, 'xlarge': 0.0}
    {'large': 0.0, 'medium': 0.0, 'small': 61.0, 'tiny': 96.0, 'xlarge': 0.0}
kit only  ['PERF-5', 'HEX-2'] purity 0.5667 uncertainty 0.09
    {'large': 8.0, 'medium': 27.3, 'small': 41.3, 'tiny': 0.0, 'xlarge': 22.0}
    {'large': 0.0, 'medium': 15.7, 'small': 24.2, 'tiny': 0.0, 'xlarge': 0.0}
    {'large': 0.0, 'medium': 0.0, 'small': 50.5, 'tiny': 96.0, 'xlarge': 0.0}


## Emulated arm run (preflight fails closed; each action is a Merkle leaf)

In [3]:
r = TT.run(objs, k=2)
print(r['geometry_status'], '| seconds', r['seconds'], '| root', r['run_root'][:16])
for a in r['actions']: print(a['t_s'], a['action'], {k: v for k, v in a.items() if k not in ('t_s','action','leaf')})

PLACEHOLDER (config/tower.yaml MEASURE values) | seconds 165.5 | root 1ac31d53d1b83e71
14.6 fetch {'tray': 'SS-SLOT-3.2x19', 'from_storage': 'R1'}
25.5 insert {'tray': 'SS-SLOT-3.2x19', 'filter_slot': 2}
32.0 fetch {'tray': 'HEX-2', 'from_storage': 'A2'}
38.5 insert {'tray': 'HEX-2', 'filter_slot': 1}
58.5 pour {'z_mm': 470.0}
88.5 rinse {'litres': 0.75}
92.5 pull {'tray': 'SS-SLOT-3.2x19', 'filter_slot': 2}
100.4 image {'tray': 'SS-SLOT-3.2x19'}
113.1 return {'tray': 'SS-SLOT-3.2x19', 'to': 'R1'}
121.5 pull {'tray': 'HEX-2', 'filter_slot': 1}
129.4 image {'tray': 'HEX-2'}
140.2 return {'tray': 'HEX-2', 'to': 'A2'}
146.8 pull {'tray': 'TRAY', 'filter_slot': 0}
154.6 image {'tray': 'TRAY'}
165.5 return {'tray': 'TRAY', 'to': 'filter level 0'}


## A bad plan is refused

In [4]:
st = TT.initial_state(cfg, inv)
try:
    TT.emulate([('insert', {'tray': 'SCREEN-1', 'filter_slot': 2}), ('insert', {'tray': 'PERF-5', 'filter_slot': 1})], cfg, st, inv)
except TT.PreflightError as e:
    print('REFUSED:', e)

REFUSED: stack not coarse->fine top->bottom
